# 📈 Análise de Correlação e Geração de Gráficos

Este notebook é dedicado à análise estatística de correlação das variáveis do projeto. O objetivo principal é responder à pergunta central:
> **"De que maneira a flutuação mensal na proporção de cargos comissionados e o volume de adiantamentos emergenciais explicam a variabilidade dos gastos com pessoal entre as secretarias?"**

Para isso, este processo realiza:
1. A carga dos dados unificados em `base_unificada.csv`.
2. A fatorização de variáveis não numéricas (categóricas/textos) para viabilizar cálculos matemáticos.
3. O cálculo do coeficiente de correlação de Pearson de todas as variáveis em relação à nossa variável alvo (`target_instabilidade`).
4. A filtragem de variáveis com alto poder de correlação (onde o coeficiente é menor ou igual a -0.3, ou maior ou igual a 0.3).
5. A exportação do relatório estatístico detalhado para `CORRELACOES.md`.
6. A plotagem de gráficos interativos de correlação e dispersão cartesianos (eixos X e Y).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ajuste robusto de caminhos para funcionamento na raiz ou em '/src'
raiz_projeto = os.path.exists("bases")
caminho_base = "bases/base_unificada.csv" if raiz_projeto else "../bases/base_unificada.csv"
caminho_md = "CORRELACOES.md" if raiz_projeto else "../CORRELACOES.md"
caminho_grafico_corr = "grafico_correlacoes.png" if raiz_projeto else "../grafico_correlacoes.png"
caminho_grafico_disp = "grafico_dispersao_XY.png" if raiz_projeto else "../grafico_dispersao_XY.png"

print(f"Carregando base de: {os.path.abspath(caminho_base)}")
df = pd.read_csv(caminho_base, low_memory=False)
print(f"Base carregada com sucesso! Dimensões: {df.shape}")


## 🧬 Fatorização e Preparação dos Dados

Para calcular a correlação de Pearson de variáveis qualitativas (como tipo de empenho, função, escolaridade, cargo), precisamos convertê-las temporariamente em códigos numéricos. Faremos isso através do processo de **fatorização** (`pd.factorize`).

In [ ]:
# Colunas que servem apenas como IDs textuais, datas brutas ou componentes diretos da soma
exclude_cols = [
    'orgao', 'nomeEntidade', 'orgao_padronizado', 'mes_ano', 
    'dataPagamento', 'dataEmpenho', 'competencia', 'dataAdmissao', 
    'dataRescisao', 'target_instabilidade', 'gastos_pessoal_mensal'
]

vars_to_corr = [c for c in df.columns if c not in exclude_cols]

calc_df = pd.DataFrame()
calc_df['target_instabilidade'] = df['target_instabilidade']

print("Fatorizando variáveis categóricas...")
for col in vars_to_corr:
    if not pd.api.types.is_numeric_dtype(df[col]):
        # Se for qualitativa, fatoriza (ex: 'EFETIVO' -> 0, 'COMISSIONADO' -> 1)
        calc_df[col] = pd.factorize(df[col].astype(str))[0]
    else:
        calc_df[col] = df[col].copy()

print(f"Matriz para correlação pronta com {calc_df.shape[1]} colunas.")

## 🧮 Cálculo do Coeficiente de Pearson

Agora calcularemos a correlação e filtraremos apenas as variáveis com correlação estatisticamente relevante (coeficiente $\ge 0.3$ ou $\le -0.3$).

In [ ]:
cr = calc_df.corr()
target_cr = cr['target_instabilidade'].sort_values(ascending=False).dropna()

# Aplicando filtro de correlação forte
valid_threshold = target_cr[(target_cr >= 0.3) | (target_cr <= -0.3)]

# Removemos o 'vol_adiantamentos_mensal' da análise por ser um componente direto do target
valid_threshold = valid_threshold.drop('vol_adiantamentos_mensal', errors='ignore')

print(f"================ CORRELAÇÕES FORTES ENCONTRADAS ({len(valid_threshold)} VARIÁVEIS) ================")
print(valid_threshold.to_string())

## 💾 Exportação do Relatório para Markdown

Salvaremos os resultados em um arquivo Markdown chamado `CORRELACOES.md` na raiz do projeto.

In [ ]:
print(f"Salvando relatório em: {os.path.abspath(caminho_md)}")
with open(caminho_md, 'w', encoding='utf-8') as f:
    f.write(f"# 📊 Relatório Estatístico de Correlações\n\n")
    f.write(f"Foram identificadas **{len(valid_threshold)} variáveis** com coeficiente de correlação de Pearson forte ($|r| \\ge 0.3$) em relação à variável alvo (`target_instabilidade`).\n\n")
    f.write("| Variável | Coeficiente de Pearson (r) |\n")
    f.write("| :--- | :---: |\n")
    for var, val in valid_threshold.items():
        f.write(f"| `{var}` | {val:.6f} |\n")
        
print("✅ Relatório exportado com sucesso!")

## 📊 Gráficos de Visualização

### 1. Gráfico de Barras Horizontal das Correlações

In [ ]:
plt.figure(figsize=(12, 10))
sns.set_theme(style="whitegrid")

data_plot = valid_threshold.reset_index()
data_plot.columns = ['Variável', 'Correlação']
data_plot = data_plot.sort_values('Correlação', ascending=True)

# Criando o gráfico de barras horizontal
ax = sns.barplot(x='Correlação', y='Variável', data=data_plot, palette='vlag')

# Adicionando linhas pontilhadas de limite de correlação
plt.axvline(x=0.3, color='red', linestyle='--', linewidth=1.2, label='Corte Positivo (+0.3)')
plt.axvline(x=-0.3, color='blue', linestyle='--', linewidth=1.2, label='Corte Negativo (-0.3)')
plt.axvline(x=0, color='black', linewidth=1)

plt.title('Variáveis com Alto Poder de Correlação à Instabilidade dos Gastos', fontsize=14, pad=15)
plt.xlabel('Grau de Correlação de Pearson (r)', fontsize=12)
plt.ylabel('Variáveis (Fatorizadas/Numéricas)', fontsize=12)
plt.legend(loc='lower right')

plt.tight_layout()
plt.savefig(caminho_grafico_corr, dpi=300)
print(f"✅ Gráfico salvo em: {os.path.abspath(caminho_grafico_corr)}")
plt.show()

### 2. Plano Cartesiano (Dispersão X-Y) com Linha de Regressão

Visualizaremos a relação exata entre a **Proporção de Cargos Comissionados (Eixo X)** e a **Instabilidade Financeira Mensal (Eixo Y)**.

In [ ]:
plt.figure(figsize=(10, 8))

# Filtrando nulos e atípicos
scatter_data = df[['proporcao_comissionado_mensal', 'target_instabilidade']].dropna()

# Renderizando gráfico de dispersão com reta de regressão linear
sns.regplot(
    data=scatter_data, 
    x='proporcao_comissionado_mensal', 
    y='target_instabilidade',
    scatter_kws={'alpha': 0.35, 'color': 'darkcyan', 's': 30},
    line_kws={'color': 'firebrick', 'linewidth': 2.5, 'label': 'Linha de Tendência'}
)

plt.title('Plano Cartesiano: Instabilidade de Gastos vs Proporção de Comissionados', fontsize=14, pad=15)
plt.xlabel('Eixo X: Proporção de Servidores Comissionados (%)', fontsize=12)
plt.ylabel('Eixo Y: Instabilidade Financeira Mensal (Impacto)', fontsize=12)
plt.legend(loc='upper right')

plt.tight_layout()
plt.savefig(caminho_grafico_disp, dpi=300)
print(f"✅ Gráfico cartesiano salvo em: {os.path.abspath(caminho_grafico_disp)}")
plt.show()